In [1]:
import coflandscaper as cl

/Users/gregorlauter/Documents/PhD/2d-cof-mapper/COF-Landscaper/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


### Before you run this notebook (required)

- Place exactly one node XYZ in 0_node/ and one linker XYZ in 0_linker/.
- At the connection points, insert a dummy atom: **He** if node and linker are connected by a single bond and **Se** f node and linker are connected by a double bond .
- If your COF contains Se atoms, this workflow will not work with double bonds.
- This workflow is in general not tested for metal-containing COFs or MOFs.
- Choose a unique `COF_NAME`; all outputs will be saved in a folder named after the COF.

### Settings & options (what you can choose)

- `TOPOLOGY`: "hcb" or "sql".
- `BOND_TYPE`: "single" or "double" (must match your dummy atom choice).
- `COF_NAME`: "example-cof".
- `MODE`: choose what stacking modes to investigate: "incl" (inclined), "serr" (serrated), or "both".

In [2]:
TOPOLOGY = "hcb"
BOND_TYPE = "single" 
COF_NAME = "cof-1"
MODE = 'both'

### Build single-layer COF + Pre-Optimization:

Creates the single-layer COF and then pre-optimizes it with MACE.


In [4]:
builder = cl.BuildCOF2D()
builder.build(topo=TOPOLOGY, bond_type=BOND_TYPE, cof_name=COF_NAME)


>>> == Min RMSD of (node type: 0, node bb: boronate_ester): 9.36E-05
>>> Pre-location at node slot 0, (node type: 0, node bb: boronate_ester), RMSD: 9.36E-05
>>> Pre-location at node slot 1, (node type: 0, node bb: boronate_ester), RMSD: 9.36E-05
>>> Topology optimization starts.
>>> MESSAGE: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
>>> SUCCESS: True
>>> ITER: 10
>>> OBJ: 0.000
>>> Location at node slot 0, (node type: 0, node bb: boronate_ester), RMSD: 8.04E-05
>>> Location at node slot 1, (node type: 0, node bb: boronate_ester), RMSD: 8.29E-05
>>> Start placing edges.
/Users/gregorlauter/Documents/PhD/2d-cof-mapper/COF-Landscaper/.venv/lib/python3.12/site-packages/scipy/spatial/transform/_rotation.py:2586: UserWarning: Optimal rotation is not uniquely or poorly defined for the given sets of vectors.
  q, rssd, sensitivity = backend.align_vectors(a, b, weights, return_sensitivity)
>>> Start finding bonds in generated framework.
>>> Start finding bonds in building blocks.
>>

['cof-1/1_cof-1_single_layer/cof-1_unopt.cif']

In [6]:
preopt = cl.MacePreopt(fmax=0.01, head="omol")
preopt.run(cof_name=COF_NAME)

       Step     Time          Energy          fmax
LBFGS:    0 10:13:49   -35219.691403        2.531200
LBFGS:    1 10:13:50   -35220.417714        0.979380
LBFGS:    2 10:13:51   -35220.625777        0.979075
LBFGS:    3 10:13:51   -35220.705603        0.918868
LBFGS:    4 10:13:52   -35220.863534        0.674890
LBFGS:    5 10:13:52   -35220.897847        0.629527
LBFGS:    6 10:13:53   -35221.012863        0.493891
LBFGS:    7 10:13:53   -35221.131051        0.501316
LBFGS:    8 10:13:54   -35221.301498        0.579809
LBFGS:    9 10:13:54   -35221.466246        0.596610
LBFGS:   10 10:13:55   -35221.620188        0.583492
LBFGS:   11 10:13:56   -35221.756774        0.578906
LBFGS:   12 10:13:56   -35221.882081        0.588035
LBFGS:   13 10:13:57   -35222.017238        0.582081
LBFGS:   14 10:13:57   -35222.172470        0.561781
LBFGS:   15 10:13:58   -35222.332796        0.516512
LBFGS:   16 10:13:58   -35222.422985        0.268068
LBFGS:   17 10:13:59   -35222.437869        0.09

### Create ILD×ILS structure-matrix

Generate stacking variants by changing interlayer distance (ILD, along $z$) and interlayer slipping (ILS, in-plane).

Defaults:
- Default max. slip length/angle correspond to AB stacking and are auto-computed.
- Default ILD range is 3.0 to 4.5 in 0.1 A steps.
- Default ILS range is 0 to max in 1 A steps.

In [ ]:
matrix = cl.CreateMatrix()
matrix.run(cof_name=COF_NAME, topo=TOPOLOGY, mode=MODE)

### MACE single-point energies

Compute locally on this machine single-point energies for each structure to build the energy landscape. No geometry optimization is performed.

In [ ]:
sp = cl.MaceSP()
sp.run_mode(cof_name=COF_NAME, mode=MODE)

### Potential Energy Landscape (PES)

The PES is an approximation that treats each structure as identical except for its `ILD`/`ILS` values. Under this assumption, the landscape provides a reasonable qualitative map of relative energies across stacking configurations. The goal is to identify candidate minima that can then be validated and refined with full geometry optimization.

In [ ]:
landscape = cl.Landscape()
landscape.run_mode(cof_name=COF_NAME, mode=MODE)

Add extra (ILD, ILS) points you want to include in the selection. Use a list of tuples in Angstrom: [(ILD, ILS), ...]. These are added on top of the automatically detected minima for each mode (serr/incl).

In [ ]:
EXTRA_SERR = [(3.6, 0.0)]
EXTRA_INCL = [(3.6, 0.0)]

Selection copies CIFs at the auto-detected minima (and any extra ILD/ILS pairs you provide) into the selection folders for downstream optimization.

In [ ]:
selector = cl.SelectCofs()
selector.run_mode(cof_name=COF_NAME, mode=MODE, selections_serr=EXTRA_SERR, selections_incl=EXTRA_INCL)

### MACE geometry optimization

Perform MACE relaxations of the selected stacking structures to locate locally optimized geometries and refine relative energies prior to higher-level validation.

In [ ]:
# Defaults
opt = cl.MaceFullOpt()
opt.run_mode(cof_name=COF_NAME, mode=MODE)

### Analyze + Visualize

`analyze` computes ILD/ILS for the final optimized structures and writes `final_structures.csv` in the output folder (defaults to `COF_NAME`/4_{`COF_NAME`}_final_structures).

In [ ]:
# Defaults
analyze = cl.analyze
analyze(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Defaults
visualize = cl.visualizecof
visualize(cof_name=COF_NAME, mode=MODE)